In [3]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib as plt
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance


# Load data
df = pd.read_csv("Spotify_Dataset_V3.csv", sep = ";")
# Transform the semi-string columns to only numbers
df["# of Artist"] = df["# of Artist"].str.extract(r'(\d+)').astype(float)
df["# of Nationality"] = df["# of Nationality"].str.extract(r'(\d+)').astype(float)
# Transform the dates into only months - is there a correlation?
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y", errors="coerce")
df["Month"] = df["Date"].dt.month
# Scout data's structure. Data is clean, no need for further cleaning
target_col = "Points (Total)"
string_cols = ["Rank", "Title", "Artists", "Date", "Artist (Ind.)", "Nationality", "Continent", "Points (Ind for each Artist/Nat)", "id", "Song URL", "# of Artist", "# of Nationality"]

y = df[target_col]
X = df.drop(columns=[target_col] + string_cols)
# --- Split data ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
feature_names = X.columns.to_list()
# --- Normalize ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

df.head()

,Rank,Title,Artists,Date,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,...,# of Artist,Artist (Ind.),# of Nationality,Nationality,Continent,Points (Total),Points (Ind for each Artist/Nat),id,Song URL,Month
0,1,Ella Baila Sola,"Eslabon Armado, Peso Pluma",2023-05-29,0.668,0.758,-5176.0,0.033,0.483,0.000,...,1.0,Eslabon Armado,1.0,Mexico,Latin-America,200,100.0,3qQbCzHBycnDpGskqOWY0E,https://open.spotify.com/track/3qQbCzHBycnDpGs...,5
1,1,Ella Baila Sola,"Eslabon Armado, Peso Pluma",2023-05-29,0.668,0.758,-5176.0,0.033,0.483,0.000,...,2.0,Peso Pluma,2.0,Mexico,Latin-America,200,100.0,3qQbCzHBycnDpGskqOWY0E,https://open.spotify.com/track/3qQbCzHBycnDpGs...,5
2,2,WHERE SHE GOES,Bad Bunny,2023-05-29,0.652,0.800,-4019.0,0.061,0.143,0.629,...,1.0,Bad Bunny,1.0,Puerto Rico,Latin-America,199,199.0,7ro0hRteUMfnOioTFI5TG1,https://open.spotify.com/track/7ro0hRteUMfnOio...,5
3,3,La Bebe - Remix,"Yng Lvcas, Peso Pluma",2023-05-29,0.812,0.479,-5678.0,0.333,0.213,0.000,...,1.0,Yng Lvcas,1.0,Mexico,Latin-America,198,99.0,2UW7JaomAMuX9pZrjVpHAU,https://open.spotify.com/track/2UW7JaomAMuX9pZ...,5
4,3,La Bebe - Remix,"Yng Lvcas, Peso Pluma",2023-05-29,0.812,0.479,-5678.0,0.333,0.213,0.000,...,2.0,Peso Pluma,2.0,Mexico,Latin-America,198,99.0,2UW7JaomAMuX9pZrjVpHAU,https://open.spotify.com/track/2UW7JaomAMuX9pZ...,5


In [1]:
# ---  Linear Regressor  ---
start = time.time()

linreg = LinearRegression()
linreg.fit(X_train_scaled, y_train)
y_pred_lin = linreg.predict(X_test_scaled)

end = time.time()
elapsed = end - start
hours, rem = divmod(elapsed, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Execution time: {int(hours)}h {int(minutes)}m {seconds:.2f}s")

print("Linear Regression:")
print("  R²  =", r2_score(y_test, y_pred_lin))
print("  MAE =", mean_absolute_error(y_test, y_pred_lin))

lin_coefs = pd.Series(linreg.coef_, index=feature_names).sort_values(key=lambda s: s.abs(), ascending=False)
print("\nTop Linear Regression coefficients (on SCALED features):")
print(lin_coefs.head(15))

NameError: name 'time' is not defined

In [7]:
# ---  Non-Linear Regressor  ---
start = time.time()

degree = 4

poly_reg = Pipeline([
    ("poly_features", PolynomialFeatures(degree=degree, include_bias=False)),
    ("lin_reg", LinearRegression())
])

poly_reg.fit(X_train_scaled, y_train)
y_pred_poly = poly_reg.predict(X_test_scaled)

end = time.time()
elapsed = end - start
hours, rem = divmod(elapsed, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Execution time: {int(hours)}h {int(minutes)}m {seconds:.2f}s")

print(f"Polynomial Regression (degree = {degree}):")
print("  R²  =", r2_score(y_test, y_pred_poly))
print("  MAE =", mean_absolute_error(y_test, y_pred_poly))

Execution time: 0h 0m 14.65s
Polynomial Regression (degree = 4):
  R²  = 0.06025748904223682
  MAE = 47.48559879082672


In [ ]:
# ---  Random forest  ---
start = time.time()

rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

end = time.time()
elapsed = end - start
hours, rem = divmod(elapsed, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Execution time: {int(hours)}h {int(minutes)}m {seconds:.2f}s")

print("Random Forest:")
print("  R²  =", r2_score(y_test, y_pred_rf))
print("  MAE =", mean_absolute_error(y_test, y_pred_rf))

rf_imps = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
print("\nTop Random Forest feature importances:")
print(rf_imps.head(15))

Execution time: 0h 12m 17.23s
Random Forest:
  R²  = 0.7159365159870397
  MAE = 20.376274157781296


ValueError: Length of values (8) does not match length of index (494)

In [ ]:
# ---  Multi-Layer Perceptron (neural network) ---
start = time.time()

mlp = MLPRegressor(hidden_layer_sizes=(32, 16),
                   activation="relu",
                   max_iter=1000,
                   random_state=42)
mlp.fit(X_train_scaled, y_train)
y_pred_mlp = mlp.predict(X_test_scaled)

end = time.time()
elapsed = end - start
hours, rem = divmod(elapsed, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Execution time: {int(hours)}h {int(minutes)}m {seconds:.2f}s")

print("\nMLP Neural Network:")
print("  R²  =", r2_score(y_test, y_pred_mlp))
print("  MAE =", mean_absolute_error(y_test, y_pred_mlp))

pi = permutation_importance(
    mlp, X_test_scaled, y_test, n_repeats=10, random_state=42, n_jobs=-1
)
mlp_imps = pd.Series(pi.importances_mean, index=feature_names).sort_values(ascending=False)
print("\nTop MLP (permutation) importances:")
print(mlp_imps.head(15))

# 15:22